# Deepfake Voice Detection — ASVspoof 2019 LA CNN Trainer

Train a small CNN on the **ASVspoof 2019 Logical Access** subset and export it
to **TensorFlow.js** for in-browser inference.

**How to use**
1. Open this notebook in [Google Colab](https://colab.research.google.com/) (Runtime → GPU).
2. Run all cells top-to-bottom.
3. At the end, download `tfjs_model.zip`, unzip it, and copy the contents into
   the project at `public/model/`.

The preprocessing here MUST match `src/lib/deepfakeModel.ts` exactly:
- 16 kHz mono, trim silence, fixed 4 s
- Mel-spectrogram: `n_fft=1024`, `hop=256`, `n_mels=64`, fmin=0, fmax=8000
- log-power dB normalized to [0, 1]
- input shape `[N_MELS=64, N_FRAMES=251, 1]`


In [ ]:
!pip install -q tensorflow==2.15.0 tensorflowjs==4.17.0 librosa==0.10.1 soundfile

## 1. Download ASVspoof 2019 LA
This grabs the public Edinburgh DataShare archive (~24 GB compressed for the
full set; the LA subset is ~10 GB). On Colab Free this can take a while —
consider mounting your own Google Drive copy if you have one.

In [ ]:
import os, subprocess
os.makedirs('data', exist_ok=True)
# Mirror of ASVspoof 2019 LA hosted on Edinburgh DataShare
URL = 'https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip'
if not os.path.exists('data/LA.zip'):
    subprocess.run(['wget', '-O', 'data/LA.zip', URL], check=True)
if not os.path.exists('data/LA'):
    subprocess.run(['unzip', '-q', 'data/LA.zip', '-d', 'data/'], check=True)
print('Done.')

## 2. Load protocol files (bonafide vs spoof labels)

In [ ]:
import pandas as pd
PROTO_DIR = 'data/LA/ASVspoof2019_LA_cm_protocols'
AUDIO_DIR = 'data/LA/ASVspoof2019_LA_train/flac'

cols = ['speaker', 'filename', 'system', 'attack', 'label']
train_df = pd.read_csv(f'{PROTO_DIR}/ASVspoof2019.LA.cm.train.trn.txt',
                       sep=' ', names=cols)
print(train_df['label'].value_counts())
train_df.head()

## 3. Preprocessing — Mel-spectrogram (must match the browser)

In [ ]:
import numpy as np, librosa

SR = 16000
DUR_S = 4
N_SAMPLES = SR * DUR_S
N_FFT = 1024
HOP = 256
N_MELS = 64

def trim_silence(y, thr=0.005):
    mask = np.abs(y) > thr
    if not mask.any(): return y
    i, j = np.argmax(mask), len(mask) - np.argmax(mask[::-1])
    return y[i:j]

def pad_or_crop(y, n=N_SAMPLES):
    if len(y) >= n: return y[:n]
    out = np.zeros(n, dtype=np.float32); out[:len(y)] = y; return out

def file_to_mel(path):
    y, sr = librosa.load(path, sr=SR, mono=True)
    y = trim_silence(y)
    y = pad_or_crop(y)
    S = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
        fmin=0, fmax=SR/2, power=2.0, htk=False)
    S_db = 10 * np.log10(np.maximum(S, 1e-10))
    S_norm = np.clip((S_db + 80) / 80, 0, 1).astype(np.float32)
    # shape: [N_MELS, N_FRAMES]; frames = N_SAMPLES/HOP + 1 = 251
    return S_norm

# Smoke test
sample = train_df.iloc[0]
mel = file_to_mel(f'{AUDIO_DIR}/{sample.filename}.flac')
print(mel.shape, mel.min(), mel.max())

## 4. Build the dataset (cached on disk)

In [ ]:
import numpy as np, os
from tqdm import tqdm

CACHE = 'mel_cache.npz'
if not os.path.exists(CACHE):
    X, y = [], []
    for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
        try:
            mel = file_to_mel(f'{AUDIO_DIR}/{row.filename}.flac')
            X.append(mel)
            y.append(0 if row.label == 'bonafide' else 1)
        except Exception as e:
            print('skip', row.filename, e)
    X = np.stack(X)[..., None]   # [N, 64, 251, 1]
    y = np.array(y, dtype=np.float32)
    np.savez_compressed(CACHE, X=X, y=y)
data = np.load(CACHE)
X, y = data['X'], data['y']
print(X.shape, y.shape, 'fake ratio:', y.mean())

## 5. Train the CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.1,
                                          stratify=y, random_state=42)

def build_model(input_shape):
    m = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, padding='same'), layers.BatchNormalization(),
        layers.ReLU(), layers.MaxPool2D(2),

        layers.Conv2D(64, 3, padding='same'), layers.BatchNormalization(),
        layers.ReLU(), layers.MaxPool2D(2),

        layers.Conv2D(128, 3, padding='same'), layers.BatchNormalization(),
        layers.ReLU(), layers.MaxPool2D(2),

        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return m

model = build_model(X.shape[1:])
model.summary()

# Class weights — ASVspoof LA is heavily skewed toward spoof
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_tr)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print('class_weight:', class_weight)

hist = model.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                 epochs=15, batch_size=32, class_weight=class_weight)

## 6. Export to TensorFlow.js

In [ ]:
import tensorflowjs as tfjs, shutil
model.save('model.h5')
shutil.rmtree('tfjs_model', ignore_errors=True)
tfjs.converters.save_keras_model(model, 'tfjs_model')
!ls -lh tfjs_model
shutil.make_archive('tfjs_model', 'zip', 'tfjs_model')
print('Created tfjs_model.zip — download it from the Files panel.')

## 7. Drop into the project
Unzip `tfjs_model.zip` and copy the files into `public/model/`:

```
public/model/
├── model.json
└── group1-shard1of1.bin
```

Refresh the app — the next time you analyze audio, the **real CNN** will run
fully in your browser, no server needed.
